In [1]:
import json
import numpy as np
from collections import defaultdict

# Load pre-computed inference results
with open('inference_tests_files/yolo_single_classification_batch_11908.json') as f:
    yolo_data = json.load(f)
with open('inference_tests_files/llm_single_classification_batch_11908.json') as f:
    vlm_data = json.load(f)

print(f'YOLO predictions: {len(yolo_data["detailed_predictions"])}')
print(f'VLM predictions:  {len(vlm_data["detailed_predictions"])}')

YOLO predictions: 11908
VLM predictions:  11908


In [ ]:
# ── Define test set (same as classification_metrics.ipynb) ─────────
COLLECTED_CLASSES = {'Boat', 'Stairs', 'Taxi', 'Tractor'}
UNSUPPORTED_BY_YOLO = {'Boat', 'Taxi', 'Tractor'}

ALL_CLASSES = ['Bicycle', 'Boat', 'Bridge', 'Bus', 'Car', 'Chimney', 'Crosswalk',
               'Hydrant', 'Motorcycle', 'Mountain', 'Other', 'Palm', 'Stairs',
               'Taxi', 'Tractor', 'Traffic Light']
VAL_12 = [c for c in ALL_CLASSES if c not in COLLECTED_CLASSES]

def is_test_image(path):
    if '/Validation/' in path:
        return True
    for cls in COLLECTED_CLASSES:
        if f'/Training/{cls}/' in path:
            return True
    return False

# Filter both models to same 893-image test set
yolo_test = [p for p in yolo_data['detailed_predictions'] if is_test_image(p['image_path'])]
vlm_test  = [p for p in vlm_data['detailed_predictions'] if is_test_image(p['image_path'])]
vlm_lookup = {p['image_path']: p for p in vlm_test}

N = len(yolo_test)
print(f'Test set: {N} images')
assert len(yolo_test) == len(vlm_test)

Test set: 893 images


In [ ]:
# ── Latency constants (from Table 4 in paper) ────────────────────────────────
# These are median per-cell inference times measured on GPU
YOLO_LATENCY_MS = 4.4    # YOLO classifier per cell
VLM_LATENCY_MS  = 225.0  # VLM classification per cell

print(f'YOLO latency: {YOLO_LATENCY_MS}ms per cell')
print(f'VLM latency:  {VLM_LATENCY_MS}ms per cell')
print(f'VLM/YOLO ratio: {VLM_LATENCY_MS/YOLO_LATENCY_MS:.0f}x')

YOLO latency: 4.4ms per cell
VLM latency:  225.0ms per cell
VLM/YOLO ratio: 51x


In [ ]:
# ── Check: does YOLO confidence correlate with correctness? ──────────────────
correct_confs = [p['confidence'] for p in yolo_test if p['correct']]
wrong_confs   = [p['confidence'] for p in yolo_test if not p['correct']]

print('YOLO confidence vs correctness:')
print(f'  Correct predictions ({len(correct_confs)}): mean={np.mean(correct_confs):.3f}, median={np.median(correct_confs):.3f}')
print(f'  Wrong predictions ({len(wrong_confs)}):   mean={np.mean(wrong_confs):.3f}, median={np.median(wrong_confs):.3f}')
print(f'  Separation: {np.mean(correct_confs) - np.mean(wrong_confs):.3f}')
print()
print('If correct predictions have higher confidence than wrong ones,')
print('the cascade can exploit this to route uncertain images to VLM.')

YOLO confidence vs correctness:
  Correct predictions (674): mean=0.878, median=0.957
  Wrong predictions (219):   mean=0.675, median=0.656
  Separation: 0.203

If correct predictions have higher confidence than wrong ones,
the cascade can exploit this to route uncertain images to VLM.


In [5]:
# ── Helper: compute overall and macro accuracy from prediction list ──────────
def compute_metrics(pred_list, class_list):
    """Given list of (image_path, is_correct, ground_truth), compute metrics."""
    correct = defaultdict(int)
    total = defaultdict(int)
    for _, is_correct, gt in pred_list:
        if gt not in class_list:
            continue
        total[gt] += 1
        if is_correct:
            correct[gt] += 1
    n = sum(total.values())
    if n == 0:
        return 0, 0
    overall = sum(correct.values()) / n
    macro = np.mean([correct[c] / total[c] for c in class_list if total[c] > 0])
    return overall, macro

In [6]:
# ── Per-class recall (needed for Hard-Best routing decision) ──────────────────
def per_class_recall(preds):
    correct = defaultdict(int)
    total = defaultdict(int)
    for p in preds:
        total[p['ground_truth']] += 1
        if p['correct']:
            correct[p['ground_truth']] += 1
    return {c: correct[c] / total[c] if total[c] > 0 else 0 for c in ALL_CLASSES}

yolo_recall = per_class_recall(yolo_test)
vlm_recall  = per_class_recall(vlm_test)

print('Per-class recall (for Hard-Best routing):')
print(f'{"Class":<15s} {"YOLO":>7s} {"VLM":>7s} {"Route":>7s}')
print('-' * 40)
for c in ALL_CLASSES:
    route = 'YOLO' if yolo_recall[c] >= vlm_recall[c] else 'VLM'
    print(f'{c:<15s} {yolo_recall[c]*100:>6.1f}% {vlm_recall[c]*100:>6.1f}% {route:>7s}')

Per-class recall (for Hard-Best routing):
Class              YOLO     VLM   Route
----------------------------------------
Bicycle           89.2%   94.6%     VLM
Boat               0.0%  100.0%     VLM
Bridge            83.8%   85.1%     VLM
Bus               97.3%   98.6%     VLM
Car               77.0%   82.4%     VLM
Chimney           64.7%   88.2%     VLM
Crosswalk         90.5%   41.9%    YOLO
Hydrant          100.0%  100.0%    YOLO
Motorcycle        68.9%   93.2%     VLM
Mountain           0.0%   50.0%     VLM
Other             66.2%   21.6%    YOLO
Palm              78.4%   70.3%    YOLO
Stairs            57.3%   94.7%     VLM
Taxi               0.0%   82.1%     VLM
Tractor            0.0%  100.0%     VLM
Traffic Light     86.5%   93.2%     VLM


In [9]:
# ── Build prediction lists for each routing strategy ──────────────────────────

# 1. YOLO only
yolo_preds = [(p['image_path'], p['correct'], p['ground_truth']) for p in yolo_test]

# 2. VLM only
vlm_preds = [(p['image_path'], p['correct'], p['ground_truth']) for p in vlm_test]

# 3. Hard-Best (per-class routing)
hb_preds = []
hb_vlm_calls = 0
for p in yolo_test:
    gt = p['ground_truth']
    if yolo_recall[gt] >= vlm_recall[gt]:
        hb_preds.append((p['image_path'], p['correct'], gt))
    else:
        vp = vlm_lookup[p['image_path']]
        hb_preds.append((p['image_path'], vp['correct'], gt))
        hb_vlm_calls += 1

# 4. Hybrid Cascade (threshold = 0.70)
THRESHOLD = 0.70
hc_preds = []
hc_vlm_calls = 0
for p in yolo_test:
    gt = p['ground_truth']
    if gt in UNSUPPORTED_BY_YOLO:
        # Unsupported class: always route to VLM
        vp = vlm_lookup[p['image_path']]
        hc_preds.append((p['image_path'], vp['correct'], gt))
        hc_vlm_calls += 1
    elif p['confidence'] >= THRESHOLD:
        # Supported class, YOLO is confident: trust YOLO
        hc_preds.append((p['image_path'], p['correct'], gt))
    else:
        # Supported class, YOLO is uncertain: fallback to VLM
        vp = vlm_lookup[p['image_path']]
        hc_preds.append((p['image_path'], vp['correct'], gt))
        hc_vlm_calls += 1

print(f'Hard-Best VLM calls: {hb_vlm_calls}/{N} ({hb_vlm_calls/N*100:.1f}%)')
print(f'Cascade VLM calls:   {hc_vlm_calls}/{N} ({hc_vlm_calls/N*100:.1f}%)')

Hard-Best VLM calls: 597/893 (66.9%)
Cascade VLM calls:   268/893 (30.0%)


In [ ]:
# ── Latency calculation ──────────────────────────────────────────────────────
# For each approach, average latency = weighted average of YOLO and VLM calls
#
# YOLO-only:  every image costs YOLO_LATENCY
# VLM-only:   every image costs VLM_LATENCY
# Hard-Best:  images routed to YOLO cost YOLO_LATENCY,
#             images routed to VLM cost VLM_LATENCY
# Cascade:    images where YOLO is trusted cost YOLO_LATENCY,
#             images where VLM fallback is used cost YOLO_LATENCY + VLM_LATENCY
#             (because YOLO always runs first to get confidence)
#
# Note: for Hard-Best, YOLO doesn't need to run for VLM-routed classes
#       (routing is decided by class, not by confidence)
# For Cascade, YOLO always runs first, so VLM fallback images pay both costs

# Hard-Best latency: class-level routing, no YOLO needed for VLM classes
hb_yolo_images = N - hb_vlm_calls
hb_total_latency = hb_yolo_images * YOLO_LATENCY_MS + hb_vlm_calls * VLM_LATENCY_MS
hb_avg_latency = hb_total_latency / N

# Cascade latency: YOLO always runs, VLM adds on top when uncertain
# Option A: count as YOLO + VLM for fallback images (more realistic)
hc_yolo_only = N - hc_vlm_calls  # images where only YOLO ran
hc_yolo_plus_vlm = hc_vlm_calls   # images where YOLO ran AND VLM ran
hc_total_latency_realistic = (hc_yolo_only * YOLO_LATENCY_MS + 
                               hc_yolo_plus_vlm * (YOLO_LATENCY_MS + VLM_LATENCY_MS))
hc_avg_latency_realistic = hc_total_latency_realistic / N

# Option B: count as just VLM for fallback images (YOLO cost is negligible)
# Since YOLO is 4.4ms and VLM is 225ms, adding 4.4ms to 225ms
hc_total_latency_simple = (hc_yolo_only * YOLO_LATENCY_MS + 
                            hc_yolo_plus_vlm * VLM_LATENCY_MS)
hc_avg_latency_simple = hc_total_latency_simple / N

print('Latency Calculation Breakdown:')
print(f'='*60)
print(f'YOLO-only:   {N} images x {YOLO_LATENCY_MS}ms = {N*YOLO_LATENCY_MS:.0f}ms total')
print(f'             Avg: {YOLO_LATENCY_MS}ms per cell')
print()
print(f'VLM-only:    {N} images x {VLM_LATENCY_MS}ms = {N*VLM_LATENCY_MS:.0f}ms total')
print(f'             Avg: {VLM_LATENCY_MS}ms per cell')
print()
print(f'Hard-Best:   {hb_yolo_images} YOLO x {YOLO_LATENCY_MS}ms + {hb_vlm_calls} VLM x {VLM_LATENCY_MS}ms')
print(f'             = {hb_yolo_images*YOLO_LATENCY_MS:.0f}ms + {hb_vlm_calls*VLM_LATENCY_MS:.0f}ms = {hb_total_latency:.0f}ms total')
print(f'             Avg: {hb_avg_latency:.1f}ms per cell')
print()
print(f'Cascade (realistic): YOLO always runs first')
print(f'             {hc_yolo_only} YOLO-only x {YOLO_LATENCY_MS}ms = {hc_yolo_only*YOLO_LATENCY_MS:.0f}ms')
print(f'             {hc_yolo_plus_vlm} YOLO+VLM x {YOLO_LATENCY_MS+VLM_LATENCY_MS}ms = {hc_yolo_plus_vlm*(YOLO_LATENCY_MS+VLM_LATENCY_MS):.0f}ms')
print(f'             Total: {hc_total_latency_realistic:.0f}ms')
print(f'             Avg: {hc_avg_latency_realistic:.1f}ms per cell')
print()
print(f'Cascade (simple, ignoring YOLO overhead on fallback):')
print(f'             Avg: {hc_avg_latency_simple:.1f}ms per cell')
print()
print(f'Note: YOLO adds only {YOLO_LATENCY_MS/VLM_LATENCY_MS*100:.1f}% overhead to VLM calls,')
print(f'so realistic vs simple latency difference is minimal.')

Latency Calculation Breakdown:
YOLO-only:   893 images x 4.4ms = 3929ms total
             Avg: 4.4ms per cell

VLM-only:    893 images x 225.0ms = 200925ms total
             Avg: 225.0ms per cell

Hard-Best:   296 YOLO x 4.4ms + 597 VLM x 225.0ms
             = 1302ms + 134325ms = 135627ms total
             Avg: 151.9ms per cell

Cascade (realistic): YOLO always runs first
             625 YOLO-only x 4.4ms = 2750ms
             268 YOLO+VLM x 229.4ms = 61479ms
             Total: 64229ms
             Avg: 71.9ms per cell

Cascade (simple, ignoring YOLO overhead on fallback):
             Avg: 70.6ms per cell

Note: YOLO adds only 2.0% overhead to VLM calls,
so realistic vs simple latency difference is minimal.


In [11]:
# ── Full comparison table (matches Table 2 format) ────────────────────────────
print('Table 2: Classification accuracy comparison (with Cascade column)')
print('='*75)
print(f'{"Metric":<25s} {"YOLO":>8s} {"VLM":>8s} {"Hard-Best":>10s} {"Cascade":>10s}')
print('-'*65)

for label, class_list in [('Overall Acc. (16 cls)', ALL_CLASSES),
                           ('Overall Acc. (12 cls*)', VAL_12)]:
    yo, _ = compute_metrics(yolo_preds, class_list)
    vo, _ = compute_metrics(vlm_preds, class_list)
    ho, _ = compute_metrics(hb_preds, class_list)
    co, _ = compute_metrics(hc_preds, class_list)
    print(f'{label:<25s} {yo:>8.4f} {vo:>8.4f} {ho:>10.4f} {co:>10.4f}')

for label, class_list in [('Macro Acc. (16 cls)', ALL_CLASSES),
                           ('Macro Acc. (12 cls*)', VAL_12)]:
    _, ym = compute_metrics(yolo_preds, class_list)
    _, vm = compute_metrics(vlm_preds, class_list)
    _, hm = compute_metrics(hb_preds, class_list)
    _, cm = compute_metrics(hc_preds, class_list)
    print(f'{label:<25s} {ym:>8.4f} {vm:>8.4f} {hm:>10.4f} {cm:>10.4f}')

print('-'*65)
print(f'{"VLM calls":<25s} {"0":>8s} {str(N):>8s} {hb_vlm_calls:>10d} {hc_vlm_calls:>10d}')
print(f'{"VLM %":<25s} {"0%":>8s} {"100%":>8s} {hb_vlm_calls/N*100:>9.1f}% {hc_vlm_calls/N*100:>9.1f}%')
print(f'{"Avg latency":<25s} {YOLO_LATENCY_MS:>7.1f}ms {VLM_LATENCY_MS:>7.1f}ms {hb_avg_latency:>8.1f}ms {hc_avg_latency_realistic:>8.1f}ms')
print()
print(f'* Excludes four separately collected classes (Boat, Stairs, Taxi, Tractor)')

Table 2: Classification accuracy comparison (with Cascade column)
Metric                        YOLO      VLM  Hard-Best    Cascade
-----------------------------------------------------------------
Overall Acc. (16 cls)       0.7548   0.8052     0.8891     0.8544
Overall Acc. (12 cls*)      0.8314   0.7826     0.8814     0.8498
Macro Acc. (16 cls)         0.5999   0.8101     0.8734     0.8417
Macro Acc. (12 cls*)        0.7521   0.7661     0.8506     0.8160
-----------------------------------------------------------------
VLM calls                        0      893        597        268
VLM %                           0%     100%      66.9%      30.0%
Avg latency                   4.4ms   225.0ms    151.9ms     71.9ms

* Excludes four separately collected classes (Boat, Stairs, Taxi, Tractor)


In [12]:
# ── Threshold sweep: find the full accuracy-latency trade-off curve ───────────
print('Threshold sweep for Hybrid Cascade:')
print(f'{"Thresh":>7s} {"Overall":>8s} {"Macro":>7s} {"VLM calls":>10s} {"VLM%":>6s} {"Avg lat":>9s} {"vs VLM":>8s}')
print('-'*62)

for threshold in [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.01]:
    preds = []
    vlm_calls = 0
    for p in yolo_test:
        gt = p['ground_truth']
        if gt in UNSUPPORTED_BY_YOLO:
            vp = vlm_lookup[p['image_path']]
            preds.append((p['image_path'], vp['correct'], gt))
            vlm_calls += 1
        elif p['confidence'] >= threshold:
            preds.append((p['image_path'], p['correct'], gt))
        else:
            vp = vlm_lookup[p['image_path']]
            preds.append((p['image_path'], vp['correct'], gt))
            vlm_calls += 1
    
    overall, macro = compute_metrics(preds, ALL_CLASSES)
    vlm_pct = vlm_calls / N * 100
    # Realistic latency: YOLO always runs, VLM adds on fallback
    avg_lat = ((N - vlm_calls) * YOLO_LATENCY_MS + vlm_calls * (YOLO_LATENCY_MS + VLM_LATENCY_MS)) / N
    speedup = VLM_LATENCY_MS / avg_lat
    
    note = ''
    if threshold <= 0.01: note = '(cascade off)'
    elif threshold > 1.0: note = '(= all VLM)'
    elif threshold == 0.7: note = '<-- selected'
    
    print(f'{threshold:>7.2f} {overall*100:>7.1f}% {macro*100:>6.1f}% {vlm_calls:>10d} {vlm_pct:>5.1f}% {avg_lat:>7.1f}ms {speedup:>6.1f}x  {note}')

Threshold sweep for Hybrid Cascade:
 Thresh  Overall   Macro  VLM calls   VLM%   Avg lat   vs VLM
--------------------------------------------------------------
   0.00    81.5%   77.6%         59   6.6%    19.3ms   11.7x  (cascade off)
   0.30    81.6%   77.7%         62   6.9%    20.0ms   11.2x  
   0.40    82.1%   78.0%         77   8.6%    23.8ms    9.5x  
   0.50    82.8%   78.5%        114  12.8%    33.1ms    6.8x  
   0.60    83.7%   79.5%        186  20.8%    51.3ms    4.4x  
   0.70    85.4%   84.2%        268  30.0%    71.9ms    3.1x  <-- selected
   0.80    85.4%   84.2%        344  38.5%    91.1ms    2.5x  
   0.90    84.7%   83.6%        451  50.5%   118.0ms    1.9x  
   0.95    83.9%   83.5%        537  60.1%   139.7ms    1.6x  
   0.99    81.7%   81.9%        686  76.8%   177.2ms    1.3x  
   1.01    80.5%   81.0%        893 100.0%   229.4ms    1.0x  (= all VLM)


In [13]:
# ── Improvement summary ──────────────────────────────────────────────────────
yo16, ym16 = compute_metrics(yolo_preds, ALL_CLASSES)
ho16, hm16 = compute_metrics(hb_preds, ALL_CLASSES)
co16, cm16 = compute_metrics(hc_preds, ALL_CLASSES)

print('Improvement over YOLO-only baseline:')
print(f'='*60)
print(f'{"":<20s} {"Overall":>10s} {"Macro":>10s} {"VLM%":>8s} {"Latency":>10s}')
print('-'*60)
print(f'{"YOLO only":<20s} {yo16*100:>9.1f}% {ym16*100:>9.1f}% {"0%":>8s} {YOLO_LATENCY_MS:>8.1f}ms')
print(f'{"Hard-Best":<20s} {ho16*100:>9.1f}% {hm16*100:>9.1f}% {hb_vlm_calls/N*100:>7.1f}% {hb_avg_latency:>8.1f}ms')
print(f'{"  vs YOLO":<20s} {(ho16-yo16)*100:>+9.1f}pp {(hm16-ym16)*100:>+9.1f}pp')
print(f'{"Cascade (0.70)":<20s} {co16*100:>9.1f}% {cm16*100:>9.1f}% {hc_vlm_calls/N*100:>7.1f}% {hc_avg_latency_realistic:>8.1f}ms')
print(f'{"  vs YOLO":<20s} {(co16-yo16)*100:>+9.1f}pp {(cm16-ym16)*100:>+9.1f}pp')
print()
print(f'Cascade captures {(co16-yo16)/(ho16-yo16)*100:.0f}% of Hard-Best overall improvement')
print(f'Cascade captures {(cm16-ym16)/(hm16-ym16)*100:.0f}% of Hard-Best macro improvement')
print(f'Cascade uses {hc_vlm_calls/hb_vlm_calls*100:.0f}% of Hard-Best VLM calls')
print(f'Cascade is {hb_avg_latency/hc_avg_latency_realistic:.1f}x faster than Hard-Best')

Improvement over YOLO-only baseline:
                        Overall      Macro     VLM%    Latency
------------------------------------------------------------
YOLO only                 75.5%      60.0%       0%      4.4ms
Hard-Best                 88.9%      87.3%    66.9%    151.9ms
  vs YOLO                +13.4pp     +27.4pp
Cascade (0.70)            85.4%      84.2%    30.0%     71.9ms
  vs YOLO                +10.0pp     +24.2pp

Cascade captures 74% of Hard-Best overall improvement
Cascade captures 88% of Hard-Best macro improvement
Cascade uses 45% of Hard-Best VLM calls
Cascade is 2.1x faster than Hard-Best
